# ClaimMate
An agent that helps you appeal a denied health insurance claim. It reads the denial letter, finds the buried appeal deadline, cites the **real** plan clause that supports you (or honestly **abstains** when none does), drafts the appeal, and **never finalizes without your approval**.

All data in this notebook is **synthetic**. It contains no real personal or health information.

## 1. Install

In [ ]:
!pip -q install "google-adk[db,a2a]" requests

## 2. Get the code
Replace the URL with your own repository, or attach the project files to the notebook.

In [ ]:
import os, sys, subprocess
REPO = "https://github.com/YOUR_USERNAME/claimmate.git"  # replace with your repo
if not os.path.exists("claimmate") and not os.path.exists("claimmate/claimmate"):
    try:
        subprocess.run(["git", "clone", REPO], check=True)
        os.chdir("claimmate")
    except Exception as e:
        print("Clone failed. Attach the project files instead:", e)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

## 3. Deterministic core (no API key needed)
These pieces are plain Python, so they are reliable and testable: the live ICD-10 validation, the deadline math, and the citation gate that blocks any invented clause.

In [ ]:
from claimmate import tools, guardrails, retrieval
ids = retrieval.active_index().clause_ids()
print("Plan clauses:", sorted(ids))
print("ICD-10 M54.50 ->", tools.validate_diagnosis_code("M54.50"))
print("Deadline      ->", tools.compute_appeal_deadline("2026-01-15", 180))
print("Invented clause blocked ->", guardrails.check_citations("Per §99.9 you must pay.", ids))

## 4. Add your Gemini API key
On Kaggle: **Add-ons -> Secrets -> GOOGLE_API_KEY**. Get a key from Google AI Studio.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")
except Exception:
    os.environ.setdefault("GOOGLE_API_KEY", "")  # or paste your key here
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "0"
os.environ["CLAIMMATE_DISABLE_CONFIRM"] = "1"  # notebook runs unattended
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first"

## 5. Run ClaimMate on a supported denial (MRI)
Watch the tool calls: validate the diagnosis code, compute the deadline, delegate to the Clause-Finder, validate citations.

In [ ]:
import asyncio
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from claimmate import config
from claimmate.agent import root_agent

runner = Runner(app_name=config.APP_NAME, agent=root_agent, session_service=InMemorySessionService())

async def run(letter_file, session):
    await runner.session_service.create_session(app_name=config.APP_NAME, user_id="nb", session_id=session)
    text = open(f"data/synthetic/letters/{letter_file}", encoding="utf-8").read()
    msg = types.Content(role="user", parts=[types.Part(text="Help me appeal this Northwind Health PPO denial.\n\n" + text)])
    async for ev in runner.run_async(user_id="nb", session_id=session, new_message=msg):
        c = getattr(ev, "content", None)
        if not c or not c.parts:
            continue
        for p in c.parts:
            if getattr(p, "function_call", None):
                print("  [tool]", p.function_call.name, dict(p.function_call.args or {}))
            elif getattr(p, "text", None):
                print(p.text)

await run("letter_01_mri.txt", "nb1")

## 6. The honesty beat: abstain on an unsupported (cosmetic) denial
The plan excludes this service. A naive model would invent support. ClaimMate refuses and says so.

In [ ]:
await run("letter_03_cosmetic.txt", "nb2")

## 7. Score it (eval pass-rate)
Deadline extraction, ICD-10 validation, correct clause grounding, no invented citations, and abstention on the unsupported case.

In [ ]:
!python evals/run_eval.py

## 8. Naive vs ClaimMate (the proof asset)

In [ ]:
!python scripts/before_after.py --letter data/synthetic/letters/letter_03_cosmetic.txt

## 9. Interactive UI, traces, and A2A
From the project root:
- `adk web` opens the chat UI with a live trace view (this is the observability screen).
- `uvicorn claimmate.a2a_server:app --port 8001` serves the Clause-Finder over A2A; set `CLAIMMATE_CLAUSE_FINDER_URL=http://localhost:8001` and the orchestrator delegates to it across the network.
- `adk deploy cloud_run ./claimmate` deploys the agent for a public URL (optional).